# Assessing a Classifier for Fairness Based on Movement Patterns

In [ ]:
import numpy as np
import pickle
from tqdm import tqdm
from pathlib import Path

from src.bernoulli_spatial_scan import BernoulliSpatialScan

### Set up an object modeling the Bernoulli-based spatial scan statistic

In [ ]:
# Retrieve the flattened candidates + aux info.
path_dict_candidates = './data_simulator/huge_dataset/gencand/dict_flattened_candidates.pkl' # Path to the flattened candidates.
flat_ids, indptr = BernoulliSpatialScan.load_flattened_candidates(path_dict_candidates)

First compute the Monte Carlo simulations needed to determine the distribution of the test statistics under the assumption that the null hypothesis is true. The test statistics used is the maximum log likelihood ratio computed across the regions of all the grids, while the likelihood functions to model H_0 and H_1 are the Bernoulli-based ones.

Once we have the distribution, determine if the value of the test statistics computed from the "real" labels points to unfairness somewhere or not, thus rejecting the null.

In [ ]:
# Instantiate the 'BernoulliSpatialScan' object.
num_simulations = 999   # Number of Monte Carlo simulations to derive an approx. distribution of the test statistics
alpha = 0.01            # Significance level required.
spatial_scan = BernoulliSpatialScan(num_simulations, alpha, 
                                    flat_ids, indptr)

In [ ]:
list_set_groups = ['num_hotspots', 'num_regions']
for set_groups in list_set_groups :
    print(f"Processing set of groups of datasets: '{set_groups}'")
    
    # Determine the groups of datasets to process from the files in 'path_unfair_datasets'
    path_unfair_datasets = f'./experiments/{set_groups}/'
    list_files_datasets = [f for f in Path(path_unfair_datasets).iterdir() if (f.is_file() and 'results' not in f.name)]
    # list_files_datasets


    for file_dataset in list_files_datasets :
        # Read a group of datasets from disk.
        print(f"Processing group of datasets from file {file_dataset}...")
        with open(file_dataset, "rb") as f:
            datasets = pickle.load(f)


        # Assess the considered group of unfair label datasets.
        results = {'idx_candidates' : [], 'lr_candidates' : []}
        for dataset in tqdm(datasets['data']) :

            labels = dataset[2]
            n_objects = labels.size
            positive_rate = labels.sum() / n_objects

            # reject, vec_max_LR_sims, dist_LR_labels, max_LR_labels, inrate_labels, outrate_labels = spatial_scan.sequential_simulations(labels)
            reject, vec_max_LR_sims, dist_LR_labels, max_LR_labels, inrate_labels, outrate_labels = spatial_scan.parallel_simulations(labels)

            # Find out the threshold value the log-LR of a candidate must have in order to be considered extreme.
            threshold_value = np.sort(vec_max_LR_sims)[-int(vec_max_LR_sims.size * alpha)]

            # Find out the indices of the extreme candidates, and their log-LR values.
            pos_extreme_candidates = np.flatnonzero(dist_LR_labels >= threshold_value)
            lr_extreme_candidates = dist_LR_labels[pos_extreme_candidates]

            # Append the indices of the extreme candidates found for this dataset, if any.
            results['idx_candidates'].append(pos_extreme_candidates)
            results['lr_candidates'].append(lr_extreme_candidates)


        # Write out the results for this group of datasets to disk.
        path_results = path_unfair_datasets + 'results_' + file_dataset.name
        with open(path_results, "wb") as f:
            pickle.dump(results, f)